# 01 · Train the triple-regime models

Trains the two networks compared in the manuscript and writes models, metrics and figures to `runs/notebook_01/`.

| Variant | Trained on |
|---|---|
| **noisy** | signals pooled from SNR 20, 50, 100, 150 (mixed-SNR) |
| **noise-free** | the noise-free dictionary only |

Both take the same 43-value input: the 40 L2-normalised echoes plus three R2* features (Parts A, B, C of the GESFIDE signal).

All logic lives in `src/mrvf/`; this notebook only sets paths and calls `mrvf.pipeline.run_training`.
The command-line equivalent is `python scripts/run_pipeline.py train --variant both`.

**Time:** a full run trains for up to 200 epochs per model (tens of minutes on a GPU). Set `QUICK = True` to try it in minutes.


In [ ]:
import logging, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent          # this notebook lives in <repo>/notebooks
sys.path.insert(0, str(REPO / "src"))       # lets you run without `pip install -e .`

import matplotlib.pyplot as plt
from IPython.display import Image, display

from mrvf.config import load_config, with_overrides
from mrvf.training import get_device

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)


## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────────────
DATA_DIR   = REPO.parent / "subsamples" / "subsamples_v3"   # folder with the QuasiRand_*.mat dictionaries
ECHOTIMES  = REPO.parent / "echotimes.mat"
QUICK      = False   # True: 3 epochs, 200k samples, 2 SNR levels (a few minutes, for a smoke test)
# ──────────────────────────────────────────────────────────────────────────────────────

from dataclasses import replace

cfg = load_config(REPO / "configs" / "default.toml")
cfg = replace(cfg, paths=replace(cfg.paths, dict_dir=DATA_DIR, echotimes_file=ECHOTIMES))
if QUICK:
    cfg = with_overrides(cfg, epochs=3, n_samples=200_000, snr_levels=(20, 150))

device = get_device()
print("device:", device)


## How the echoes are split

The 40 GESFIDE echoes fall into three segments; each gives one R2* estimate (slope of log-signal vs time).

In [ ]:
from mrvf.data import load_timing

timing = load_timing(cfg)
for name, t in [("Part A (FID)", timing.t_a), ("Part B (rephasing)", timing.t_b), ("Part C (post spin-echo, rel. to SE)", timing.t_c_rel)]:
    print(f"{name:<38} {len(t):2d} echoes   {t[0]*1e3:7.2f} … {t[-1]*1e3:7.2f} ms")

## Train

In [ ]:
from mrvf.pipeline import run_training

OUT = REPO / "runs" / "notebook_01"
trained = {
    variant: run_training(cfg, variant, OUT / f"triple_regime_{variant.replace('-', '_')}", device)
    for variant in ("noisy", "noise-free")
}

## Held-out test accuracy

In [ ]:
from mrvf.config import PARAM_KEYS

print(f"{'variant':<12}" + "".join(f"{p + ' RMSE':>14}" for p in PARAM_KEYS))
for variant, r in trained.items():
    print(f"{variant:<12}" + "".join(f"{r['results'][p]['rmse']:>10.3f} {r['results'][p]['unit']:<3}" for p in PARAM_KEYS))

## Figures written by the run

In [ ]:
for variant in ("noisy", "noise_free"):
    for name in ("training_curve", "scatter", "rmse_vs_snr"):
        display(Image(filename=str(OUT / f"triple_regime_{variant}" / "figures" / f"{name}.png"), width=760))

## What was written

Each variant's folder holds `models/*_best.pt` and `*_final.pt`, `training_history.json`, `test_results.json`,
`per_snr_rmse.json`, `ablation_results.json` and `figures/`.

Retraining is not bit-for-bit reproducible against the shipped `results/` checkpoints (network initialisation, batch order and
GPU kernels differ), so expect similar, not identical, numbers. Notebook 02 evaluates *checkpoints*, and reproduces the shipped
results exactly when pointed at the shipped checkpoints.